In [1]:
import pandas as pd
import os, json, librosa
import soundfile as sf
import io

In [2]:
df = pd.read_parquet(r'd:\FineTuning\train-00000-of-00002.parquet')
print(df.columns)
print(df.iloc[0])

Index(['audio', 'sentence', 'path', 'input_ids', 'labels', 'input_length',
       'speaker_embeddings'],
      dtype='str')
audio                 {'bytes': b'RIFFz\x8b\x01\x00WAVEfmt \x10\x00\...
sentence                             पञ्चकुला भारतका सहरहरू एउटा सहर हो
path                  E:/.awajai/data/slr43/ne_np_female/wavs/nep_05...
input_ids             [151, 83, 91, 110, 152, 136, 150, 137, 107, 13...
labels                [[-3.3857956, -3.1920063, -3.1041048, -2.97389...
input_length                                                      50603
speaker_embeddings    [-0.08135123, -0.022780364, 0.039687287, 0.035...
Name: 0, dtype: object


input_ids→ tokens created for another model,
labels→ likely TTS training data,
input_length→ related to previous processing,
speaker_embedding→ information about the speaker's voice,
path→ old Windows path, probably unusable in Colab

In [3]:
for i in range(10):
    audio_bytes = df.iloc[i]['audio']['bytes']
    data, sr = sf.read(io.BytesIO(audio_bytes))
    print(i, sr) # Checking whether the audio frequency is 16khz or not

0 16000
1 16000
2 16000
3 16000
4 16000
5 16000
6 16000
7 16000
8 16000
9 16000


In [4]:
# Only these two columns matter
pilot_df = df[['audio', 'sentence']].sample(n=80, random_state=42)

In [6]:
audio_dir = 'd:/FineTuning/pilot_run'
os.makedirs(audio_dir, exist_ok=True)
manifest_path = 'd:/FineTuning/pilot_manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as f:
    for i, row in pilot_df.iterrows():
        audio_bytes = row['audio']['bytes']
        wav_path = f"{audio_dir}/{i}.wav"
        data, sr = sf.read(io.BytesIO(audio_bytes))

        if sr != 16000:
          data = librosa.resample(data, orig_sr=sr, target_sr=16000)
          sr = 16000

        sf.write(wav_path, data, sr)
        duration = len(data) / sr

        entry = {
            "audio_filepath": wav_path,
            "text": row['sentence'],
            "duration": duration,
            "target_lang": "ne-NP"
        }
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [7]:
with open(manifest_path) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

{"audio_filepath": "d:/FineTuning/pilot_run/788.wav", "text": "कोलम्बियाली फुटबल प्रतियोगिता", "duration": 2.899375, "target_lang": "ne-NP"}
{"audio_filepath": "d:/FineTuning/pilot_run/440.wav", "text": "यस जिल्लालाई कालीकोट पनि भनिन्थ्यो", "duration": 2.4, "target_lang": "ne-NP"}
{"audio_filepath": "d:/FineTuning/pilot_run/139.wav", "text": "फ्रान्सेली गणतन्त्रको दोस्रो राष्ट्रपति", "duration": 2.634375, "target_lang": "ne-NP"}
{"audio_filepath": "d:/FineTuning/pilot_run/695.wav", "text": "जिल्ला आयुर्वेद स्वास्थ्यकेन्द्र पाटनले आफ्नो एकाइहरूको रूपमा ग्रामीण भेगमा शाखाहरू खोल्ने भएको छ", "duration": 7.493375, "target_lang": "ne-NP"}
{"audio_filepath": "d:/FineTuning/pilot_run/66.wav", "text": "कालीकण्ठे घाँसेफिस्टो नेपालमा पाइने एक प्रकारको चराको नाम हो", "duration": 4.754375, "target_lang": "ne-NP"}


In [9]:
with open(manifest_path, encoding='utf-8') as f:
    lines = f.readlines()

train_lines = lines[:70]
val_lines = lines[70:]

with open('d:/FineTuning/pilot_train_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(train_lines)

with open('d:/FineTuning/pilot_val_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(val_lines)

In [ ]:
with open('d:/FineTuning/pilot_train_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [ ]:
import nemo
import nemo.collections.asr as nemo_asr
print(nemo.__version__)

In [ ]:
# !git clone https://github.com/NVIDIA/NeMo.git "d:/FineTuning/NeMo"

In [ ]:
# !ls "d:/FineTuning/NeMo/examples/asr"

In [ ]:
from nemo.collections.asr.models import ASRModel

model_path = "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

model = ASRModel.restore_from(model_path)

In [ ]:
!tar -tf "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

In [ ]:
!mkdir -p d:/FineTuning/extracted_tokenizer
!tar -xf "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C d:/FineTuning/extracted_tokenizer "427ad33c6285472cb01c3eb843d2309d_tokenizer.model"
!mv "d:/FineTuning/extracted_tokenizer/427ad33c6285472cb01c3eb843d2309d_tokenizer.model" \
    "d:/FineTuning/extracted_tokenizer/tokenizer.model"
!ls -la d:/FineTuning/extracted_tokenizer

!tar -xf "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C d:/FineTuning/extracted_tokenizer "c8ead90c911846569df4738620acfc0f_vocab.txt"
!mv "d:/FineTuning/extracted_tokenizer/c8ead90c911846569df4738620acfc0f_vocab.txt" \
    "d:/FineTuning/extracted_tokenizer/vocab.txt"
!ls -la d:/FineTuning/extracted_tokenizer

In [ ]:
!tar -tf "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" | grep -i yaml

In [ ]:
!tar -xf "d:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" -C d:/FineTuning/extracted_tokenizer "model_config.yaml"

In [ ]:
import yaml

with open('d:/FineTuning/extracted_tokenizer/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)
model_cfg['tokenizer'] = {
    'dir': 'd:/FineTuning/extracted_tokenizer',
    'type': 'bpe'
}

# Point data at manifests
model_cfg['train_ds']['manifest_filepath'] = 'd:/FineTuning/pilot_train_manifest.json'
model_cfg['train_ds']['is_tarred'] = False   # using a plain manifest, not tarred shards
model_cfg['validation_ds']['manifest_filepath'] = 'd:/FineTuning/pilot_val_manifest.json'
model_cfg['test_ds']['manifest_filepath'] = 'd:/FineTuning/pilot_val_manifest.json' # It exists because the model configuration expects a test dataset configuration, even if you don't use it during training.
model_cfg["optim"]["lr"] = 0.0001
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['train_ds']['default_prompt_mode'] = 'unified'
model_cfg['train_ds']['unified_auto_ratio'] = 0.0
full_cfg = {
    'name': 'FastConformer-Transducer-BPE-Prompt-Streaming-Pilot',
    'model': model_cfg,
    'trainer': {
        'devices': 1, 'num_nodes': 1, 'max_epochs': 1,
        'accelerator': 'auto', 'precision': 'bf16',
        'log_every_n_steps': 10, 'enable_progress_bar': True,
        'num_sanity_val_steps': 0, 'enable_checkpointing': False,
        'logger': False, 'use_distributed_sampler': False,
        'max_steps': 40,
    },
    'exp_manager': {
        'exp_dir': 'd:/FineTuning/pilot_run',
        'create_tensorboard_logger': True,
        'create_checkpoint_callback': True,
        'checkpoint_callback_params': {'monitor': 'val_wer', 'mode': 'min', 'save_top_k': 1, 'always_save_nemo': True},
        'resume_if_exists': True,
        'resume_ignore_no_checkpoint':True,
    },
    'init_from_nemo_model': 'd:/FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo',
}

out_path = 'd:/FineTuning/pilot_finetune_config.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(full_cfg, f, allow_unicode=True, sort_keys=False)

print("Saved:", out_path)

In [ ]:
!python "d:/FineTuning/NeMo/examples/asr/asr_transducer/speech_to_text_rnnt_bpe_prompt.py" \
  --config-path="d:/FineTuning" \
  --config-name="pilot_finetune_config"

## Testing

In [ ]:
manifest_path = "d:/FineTuning/pilot_manifest.json"
test_manifest_path = "d:/FineTuning/test_manifest.json"

with open(manifest_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        if entry["audio_filepath"].endswith("/1025.wav"):
            with open(test_manifest_path, 'w', encoding='utf-8') as out:
                out.write(json.dumps(entry, ensure_ascii=False) + '\n')

            print(entry)
            break

In [ ]:
with open('d:/FineTuning/test_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [ ]:
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt

finetuned_model = EncDecRNNTBPEModelWithPrompt.restore_from(
    "d:/FineTuning/pilot_run/default/checkpoints/default.nemo"
)
finetuned_model = finetuned_model.cuda().eval()

In [ ]:
import lightning.pytorch as pl
from omegaconf import OmegaConf

finetuned_model.cfg.test_ds.manifest_filepath = "d:/FineTuning/test_manifest.json"
finetuned_model.setup_test_data(finetuned_model.cfg.test_ds)

trainer = pl.Trainer(accelerator="gpu", devices=1, logger=False)
trainer.test(finetuned_model)